## Download Modules

In [1]:
%pip install -q --upgrade ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -q transformers datasets pandas tqdm numpy torch "transformers[torch]" tf-keras

Note: you may need to restart the kernel to use updated packages.


## Import Modules

In [3]:
from typing import List, Any
from datasets import DatasetDict, Dataset
from transformers import (BartForConditionalGeneration, BartTokenizer, Trainer, TrainingArguments,
                          BartConfig, GenerationConfig, DataCollatorForSeq2Seq)
import pandas as pd
import math
from tqdm import tqdm
import numpy as np

2025-02-19 03:25:59.990205: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739935560.010393   25625 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739935560.016655   25625 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-19 03:26:00.037422: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
import os
print("Current Working Directory:", os.getcwd())

Current Working Directory: /home/ubuntu/citeBART/acl_200_global_rerun


In [5]:
custom_model_name = "citeBART_global_ACL200_rerun"
checkpoints_location = f"./checkpoints/{custom_model_name}"
model_save_location = f"./models/{custom_model_name}"

# Ensure directories exist
os.makedirs(checkpoints_location, exist_ok=True)
os.makedirs(model_save_location, exist_ok=True)

## Defining Functions

### Preprocessing Dataset

In [6]:
# Okay
def read_dataset():
    train_df = pd.read_csv(train_dataset_path)
    train_set = []

    for _, i in train_df.iterrows():
        temp_citing_title = i['citing_title']
        temp_citing_abstract = i['citing_abstract']
        temp_masked_context = i['masked_cit_context'].replace("OTHERCIT", "")  # "Fill the mask with an appropriate citation: " +

        temp_train_input = temp_citing_title + " </s> " + temp_citing_abstract + " </s> " + temp_masked_context

        temp_dict = {"masked_cit_context": temp_train_input, 
                     "masked_token_target": i['masked_token_target']}

        train_set.append(temp_dict)

    eval_df = pd.read_csv(eval_dataset_path)
    eval_set = []

    for _, i in eval_df.iterrows():
        temp_citing_title = i['citing_title']
        temp_citing_abstract = i['citing_abstract']
        temp_masked_context = i['masked_cit_context'].replace("OTHERCIT", "")  # "Fill the mask with an appropriate citation: " +

        temp_eval_input = temp_citing_title + " </s> " + temp_citing_abstract + " </s> " + temp_masked_context

        temp_dict = {"masked_cit_context": temp_eval_input,
                     "masked_token_target": i['masked_token_target']}

        eval_set.append(temp_dict)

    return train_set, eval_set

In [7]:
# Preprocessing function (Okay)
def preprocess_function(examples):
    inputs = [example.replace("<mask>", "<extra_id_0>", 1).replace("<mask>", "").replace("<extra_id_0>", "<mask>")
              for example in examples["masked_cit_context"]]
    targets = [example for example in examples["masked_token_target"]]

    model_inputs = tokenizer(inputs, max_length=max_token_limit, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=max_token_limit, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

### Evaluation Functions

In [8]:
# Okay
def add_spaces_after_commas(text: str) -> str:
    return ', '.join(part.strip() for part in text.split(','))  #  NEW!!!!

# Okay
def fill_mask(sentence, num_predictions):

    # Read CSV and extract relevant column as a list (Optimized)
    all_cit_list = pd.read_csv(all_citations_path)['citation_items'].tolist()

    # Tokenize input
    sentence = sentence.replace("<mask>", "<extra_id_0>", 1).replace("<mask>", "").replace("<extra_id_0>", "<mask>")
    input_ids = tokenizer.encode(sentence, return_tensors="pt", max_length=max_token_limit, truncation=True, padding="max_length").to("cuda")

    # Generate outputs
    outputs = model.generate(input_ids, generation_config=cit_generation_config)

    # Decode outputs efficiently
    predictions = [add_spaces_after_commas(tokenizer.decode(output, skip_special_tokens=True).strip()) for output in outputs]

    # Get unique predictions
    unique_predictions: List[Any] = list(dict.fromkeys(predictions))  # Remove duplicates while preserving order

    # Print the top n predictions
    # for i, pred in enumerate(unique_predictions, num_predictions):
    #     print(f"Prediction {i}: {pred} \n\n")

    # Ensure at least `num_predictions` predictions
    if len(unique_predictions) < num_predictions:
        unique_predictions.extend([unique_predictions[-1]] * (num_predictions - len(unique_predictions)))

    return unique_predictions[:num_predictions]

In [9]:
# Okay
def compare_pred_with_correct_value(predictions, ground_truth, k=10):
    """Compute Recall@k, MRR, and NDCG for a single ground truth citation."""

    recall_at_k = 0
    reciprocal_rank = 0
    ndcg = 0

    #print(f"Predictions: {predictions}")
    #print(f"Number of Predictions: {len(predictions)})
    #print(f"Ground Truth: {ground_truth}")

    # Normalize ground truth
    ground_truth = ground_truth.replace(" and ", " ").replace(" et al.,", "").replace(",", "").strip()
    truth_tokens = ground_truth.split()

    #print(f"Truth Tokens: {truth_tokens}")

    # Ensure at least two tokens for comparison
    if len(truth_tokens) < 2:
        return recall_at_k, reciprocal_rank, ndcg # No valid NDCG if no valid ground truth
    
    # Check if the ground truth appears in the top-k predictions
    for p_idx, prediction in enumerate(predictions[:k]):  # Consider only top-k predictions
        if all(token in prediction for token in truth_tokens):  # Check if any token matches
            recall_at_k = 1  # Since there's only one correct answer
            reciprocal_rank = 1 / (p_idx + 1)
            ndcg = 1 / np.log2(p_idx + 2)  # Compute DCG for rank position
            break  # No need to check further once we find the match
        
    #print(f"Recall@k: {recall_at_k}, RR: {reciprocal_rank}, NDCG: {ndcg}")

    return recall_at_k, reciprocal_rank, ndcg

In [10]:
# Okay
def calc_eval_metrics(val_dataset, k=10):
    recall_list = []
    reciprocal_rank_list = []
    ndcg_list = []
    
    write_log("=== Evaluation Started ===")  # Log start of evaluation

    for e in tqdm(val_dataset):
        masked_cit_context = e["masked_cit_context"]
        target_token = e["masked_token_target"]

        temp_predictions = fill_mask(masked_cit_context, k)  # Get top-k predictions
        
        # Compute Recall@k, MRR, and NDCG@k
        recall_at_k, temp_reciprocal_rank, temp_ndcg = compare_pred_with_correct_value(temp_predictions, target_token, k)
        
        recall_list.append(recall_at_k)
        reciprocal_rank_list.append(temp_reciprocal_rank)
        ndcg_list.append(temp_ndcg)
    
    # Compute Mean Metrics
    mean_recall_at_k = np.mean(recall_list) if recall_list else 0
    mean_reciprocal_rank_at_k = np.mean(reciprocal_rank_list) if reciprocal_rank_list else 0
    mean_ndcg_at_k = np.mean(ndcg_list) if ndcg_list else 0

    # Log final summary
    summary_log = (
        f"\n=== Final Evaluation Metrics ===\n"
        f"Mean Recall@{k}: {mean_recall_at_k:.4f}\n"
        f"Mean MRR@{k}: {mean_reciprocal_rank_at_k:.4f}\n"
        f"Mean NDCG@{k}: {mean_ndcg_at_k:.4f}\n"
        f"=============================\n"
    )
    write_log(summary_log)

    # Create a DataFrame for better readability
    metrics_data = {
        "Metric": [f"Recall@{k}", f"Mean Reciprocal Rank@{k}", f"Normalized Discounted Cumulative Gain@{k}"],
        "Value": [mean_recall_at_k, mean_reciprocal_rank_at_k, mean_ndcg_at_k]
    }
    metrics_df = pd.DataFrame(metrics_data)
    
    print("\n=======>>> Evaluation Metrics Summary\n")
    print(metrics_df.to_string(index=False))

    return metrics_df  # Optionally return metrics for logging

## Loading Dataset

In [11]:
dataset_folder = "./dataset"  # Change this to your actual dataset path
train_dataset_path = f"{dataset_folder}/cleaned_acl_global_context_dataset_train.csv"
eval_dataset_path = f"{dataset_folder}/cleaned_acl_global_context_dataset_eval.csv"

In [12]:
pretrained_model_name_or_path = "facebook/bart-base"
max_token_limit = 400

# Initialize the config
config = BartConfig.from_pretrained(pretrained_model_name_or_path, attention_dropout=0.123)

# Initialize the tokenizer
tokenizer = BartTokenizer.from_pretrained(pretrained_model_name_or_path, truncation=True,
                                          padding='max_length', model_max_length=max_token_limit)

# Set up the model
model = BartForConditionalGeneration.from_pretrained(pretrained_model_name_or_path, config=config)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

cit_generation_config = GenerationConfig.from_model_config(model.config)

cit_generation_config.max_new_tokens = 25
cit_generation_config.do_sample = False
cit_generation_config.top_k = 50
cit_generation_config.num_return_sequences = 20
cit_generation_config.early_stopping = False
cit_generation_config.num_beams = 20
cit_generation_config.forced_bos_token_id = 0

cit_generation_config.num_beam_groups = 10
cit_generation_config.diversity_penalty = 1.5

In [13]:
# Example data to view dataset structure
"""data = {
        "train": [
            {"input": "Fill the mask with an appropriate citation: models are trained end-to-end using backpropagation
             and mini-batched Adam <mask> SGD. We use dropout regularization",
             "target": "Kingma and Ba, 2014"},
            # ...
        ],
        "validation": [
            {"input": "Fill the mask with an appropriate citation: The new policy is <mask>.",
             "target": "under review."},
            # ...
        ]
    }"""

train_dataset, eval_dataset = read_dataset()

data = {
    "train": train_dataset,
    "eval": eval_dataset
}

# Convert to Dataset
train_dataset = Dataset.from_pandas(pd.DataFrame(data["train"]))
validation_dataset = Dataset.from_pandas(pd.DataFrame(data["eval"]))

dataset = DatasetDict({
    "train": train_dataset,
    "eval": validation_dataset
})

# Preprocess the datasets
tokenized_datasets = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/49584 [00:00<?, ? examples/s]

Map:   0%|          | 0/12591 [00:00<?, ? examples/s]

## Training Part

In [14]:
# Manually setting arguments (remove argparse)
custom_model_name = "citeBART_global_ACL200_rerun"
checkpoints_location = f"./checkpoints/{custom_model_name}"
model_save_location = f"./models/{custom_model_name}"

dataset_folder = "./dataset"  # Change this to your actual dataset path
train_dataset_path = f"{dataset_folder}/cleaned_acl_global_context_dataset_train.csv"
eval_dataset_path = f"{dataset_folder}/cleaned_acl_global_context_dataset_eval.csv"
all_citations_path = f"{dataset_folder}/cleaned_acl_global_citation_item_list.csv"

num_epochs = 15
warmup_steps = 500
train_and_eval_batch_sizes = 16
auto_find_batch_size_flag = True
skip_training = False

In [15]:
# Define log file location
log_file = os.path.join(model_save_location, "training_logs.txt")

# Function to log messages
def write_log(message):
    """Append messages to the log file."""
    with open(log_file, "a") as f:
        f.write(message + "\n")

# Identify the latest checkpoint if available
latest_checkpoint = None
if os.path.exists(checkpoints_location) and os.listdir(checkpoints_location):
    checkpoint_dirs = [
        os.path.join(checkpoints_location, d)
        for d in os.listdir(checkpoints_location)
        if "checkpoint" in d and os.path.isdir(os.path.join(checkpoints_location, d))
    ]
    
    if checkpoint_dirs:
        latest_checkpoint = max(checkpoint_dirs, key=os.path.getctime)

print("Latest Checkpoint:", latest_checkpoint)

Latest Checkpoint: ./checkpoints/citeBART_global_ACL200_rerun/checkpoint-92970


In [16]:
training_args = TrainingArguments(
    output_dir=checkpoints_location,
    overwrite_output_dir=True,
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    logging_strategy="epoch",
    warmup_steps=warmup_steps,
    save_strategy="epoch",
    save_total_limit=15,
    resume_from_checkpoint=latest_checkpoint is not None,
)

if auto_find_batch_size_flag:
    training_args.auto_find_batch_size = True
else:
    training_args.per_device_train_batch_size = train_and_eval_batch_sizes
    training_args.per_device_eval_batch_size = train_and_eval_batch_sizes

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["eval"],
    data_collator=data_collator,
    tokenizer=tokenizer
)

/opt/conda/lib/python3.12/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_25625/3286828109.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [17]:
import traceback

# Start training with error handling
try:
    if not skip_training:
        if latest_checkpoint:
            write_log(f"Resuming training from checkpoint: {latest_checkpoint}")
            train_output = trainer.train(resume_from_checkpoint=latest_checkpoint)
        else:
            write_log("Starting training from scratch...")
            train_output = trainer.train()

        trainer.save_model(model_save_location)
        tokenizer.save_pretrained(model_save_location)

        # Extract training metrics
        metrics = train_output.metrics
        metrics_table = pd.DataFrame([metrics])

        # Log metrics
        metrics_log = metrics_table.to_string(index=False)
        write_log("Training Completed Successfully ✅\n\nMetrics:\n" + metrics_log)

except Exception as e:
    error_message = f"Training Failed ❌\n\nError Details:\n{traceback.format_exc()}"
    
    # Log error
    write_log(error_message)

    # Re-raise the error so it doesn't silently fail
    raise

Epoch,Training Loss,Validation Loss
1,0.394500,0.021601
2,0.019700,0.015661
3,0.014700,0.013371
4,0.011800,0.011904
5,0.009700,0.011247
6,0.008200,0.010705
7,0.006900,0.010465
8,0.005900,0.010292
9,0.005000,0.010586
10,0.004300,0.010640


/opt/conda/lib/python3.12/site-packages/transformers/modeling_utils.py:2758: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


## Loading and Evaluating Model

In [ ]:
import torch

# Load Model
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Define model directory path
model_path = "models/citeBART_global_ACL200_rerun"  # Adjust this if needed

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Load model
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)


trainer = Trainer(
    model=model,
    args=training_args, # Use the same training arguments as above (rerun it)
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["eval"],
    data_collator=data_collator,
    tokenizer=tokenizer
)

print("Model and tokenizer loaded successfully!")

In [19]:
eval_results = trainer.evaluate()
print(f"\n*****************\n======>> Eval loss after fine-tuning: {eval_results['eval_loss']}\n"
      f"======>> Perplexity after fine-tuning: {math.exp(eval_results['eval_loss']):.2f}\n\n")


*****************
======>> Eval loss after fine-tuning: 0.011236312799155712
======>> Perplexity after fine-tuning: 1.01




In [21]:
# Run evaluation for epoch 15
calc_eval_metrics(eval_dataset)

100%|██████████| 12591/12591 [1:00:14<00:00,  3.48it/s]


=======>>> Evaluation Metrics Summary

                                  Metric    Value
                               Recall@10 0.680089
                 Mean Reciprocal Rank@10 0.509516
Normalized Discounted Cumulative Gain@10 0.550847


,Metric,Value
0,Recall@10,0.680089
1,Mean Reciprocal Rank@10,0.509516
2,Normalized Discounted Cumulative Gain@10,0.550847
